In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
from pathlib import Path
import re
import gc
import numpy as np
import pandas as pd
import pyarrow.dataset as ds

# =========================================================
# 0) 설정
# =========================================================
BASE_TEMP_DIR = Path(r"C:/Users/qkrtl/10th/00_Project/04_final/02_parquet_file/04_temp_parquet_files")

# 분석 대상 맵 (예: "Erangel", "Miramar", "Rondo", "Sanhok", "Taego")
TARGET_MAP = "Erangel"

# 폴더명 날짜 기준 범위
DATE_MIN = "20260212"
DATE_MAX = "20260227"

# 플랫폼 선택
PLATFORMS = {"kakao", "steam"}   # {"kakao"} / {"steam"} / {"kakao", "steam"}

# 성능 옵션
BATCH_SIZE = 65536
MAX_FILES = None      # 디버깅용: 예) 20 / 전체면 None
ASOF_TOLERANCE = "15s"

# 디버깅/메모리 옵션
VERBOSE = True

# =========================================================
# 1) temp parquet 파일 수집 (폴더명에서 platform/date 추출)
# =========================================================
folder_pat = re.compile(r"^temp_parquet_files_(kakao|steam)_(\d{8})$", re.IGNORECASE)
file_pat = re.compile(r"^(?P<map>[^_]+)_(?P<matchid>[0-9a-fA-F-]+)\.parquet$")

targets = []

for subdir in BASE_TEMP_DIR.iterdir():
    if not subdir.is_dir():
        continue
    m = folder_pat.match(subdir.name)
    if not m:
        continue

    platform, ymd = m.group(1).lower(), m.group(2)
    if platform not in PLATFORMS:
        continue
    if not (DATE_MIN <= ymd <= DATE_MAX):
        continue

    for p in subdir.glob("*.parquet"):
        fm = file_pat.match(p.name)
        if not fm:
            continue
        map_name = fm.group("map")
        if map_name != TARGET_MAP:
            continue

        targets.append({
            "path": p,
            "platform": platform,
            "src_date": ymd,            # 배치/수집 날짜
            "map": map_name,
            "file_matchid": fm.group("matchid"),
        })

targets = sorted(targets, key=lambda x: (x["src_date"], x["platform"], x["path"].name))
if MAX_FILES is not None:
    targets = targets[:MAX_FILES]

print(f"[INFO] Collected files: {len(targets)}")
if targets:
    print("[INFO] Example target:", targets[0])

# =========================================================
# 2) 필요한 이벤트/컬럼 정의
# =========================================================
EVENTS = {
    "LogParachuteLanding",
    "LogPlayerPosition",
    "LogGameStatePeriodic",
    "LogVehicleLeave",      # 없으면 fallback로 position 차량상태 사용
    "LogPhaseChange",       # 현재 간이 rotation에서는 직접 사용 안 하지만 로드 유지 가능
}

NEEDED_COLS = {
    # 공통
    "matchId", "_D", "_T", "common_isGame",

    # 공통 character
    "character_accountId", "character_name",
    "character_location_x", "character_location_y", "character_location_z",
    "distance",

    # position
    "elapsedTime", "character_teamId",
    "character_isInVehicle", "character_isInBlueZone",

    # gamestate
    "gameState_elapsedTime",
    "gameState_safetyZonePosition_x", "gameState_safetyZonePosition_y", "gameState_safetyZoneRadius",
    "gameState_poisonGasWarningPosition_x", "gameState_poisonGasWarningPosition_y", "gameState_poisonGasWarningRadius",

    # vehicle leave
    "vehicle_location_x", "vehicle_location_y", "vehicle_location_z",
    "rideDistance", "maxSpeed",

    # phase
    "phase",
}

# =========================================================
# 3) raw parquet -> 이벤트별 DataFrame 누적
# =========================================================
landing_parts = []
pos_parts = []
gs_parts = []
veh_leave_parts = []
phase_parts = []

for i, t in enumerate(targets, start=1):
    path = t["path"]
    if VERBOSE and (i % 50 == 0 or i == 1):
        print(f"[LOAD] {i}/{len(targets)} : {path.name}")

    try:
        dataset = ds.dataset(str(path), format="parquet")
    except Exception as e:
        print(f"[SKIP] dataset load failed: {path.name} | {e}")
        continue

    schema_cols = set(dataset.schema.names)
    if "_T" not in schema_cols:
        print(f"[SKIP] _T not found: {path.name}")
        continue

    cols_to_read = [c for c in NEEDED_COLS if c in schema_cols]

    try:
        scanner = dataset.scanner(
            columns=cols_to_read,
            filter=ds.field("_T").isin(list(EVENTS)),
            batch_size=BATCH_SIZE
        )
    except Exception as e:
        print(f"[SKIP] scanner failed: {path.name} | {e}")
        continue

    for batch in scanner.to_batches():
        df = batch.to_pandas()
        if df.empty:
            continue

        # 메타 정보 추가 (이후 drop_time과 src_date 비교 가능)
        df["src_platform"] = t["platform"]
        df["src_date"] = t["src_date"]
        df["src_map"] = t["map"]
        df["src_file_matchid"] = t["file_matchid"]

        if "_T" not in df.columns:
            continue

        for ev, sub in df.groupby("_T"):
            # copy() 해두는 게 안전
            if ev == "LogParachuteLanding":
                landing_parts.append(sub.copy())
            elif ev == "LogPlayerPosition":
                pos_parts.append(sub.copy())
            elif ev == "LogGameStatePeriodic":
                gs_parts.append(sub.copy())
            elif ev == "LogVehicleLeave":
                veh_leave_parts.append(sub.copy())
            elif ev == "LogPhaseChange":
                phase_parts.append(sub.copy())

    del dataset
    gc.collect()

print("[INFO] Loaded parts counts:",
      f"landing={len(landing_parts)}",
      f"pos={len(pos_parts)}",
      f"gs={len(gs_parts)}",
      f"veh_leave={len(veh_leave_parts)}",
      f"phase={len(phase_parts)}")

# concat
landing = pd.concat(landing_parts, ignore_index=True) if landing_parts else pd.DataFrame()
pos = pd.concat(pos_parts, ignore_index=True) if pos_parts else pd.DataFrame()
gs = pd.concat(gs_parts, ignore_index=True) if gs_parts else pd.DataFrame()
veh_leave = pd.concat(veh_leave_parts, ignore_index=True) if veh_leave_parts else pd.DataFrame()
phase = pd.concat(phase_parts, ignore_index=True) if phase_parts else pd.DataFrame()

# 메모리 해제
del landing_parts, pos_parts, gs_parts, veh_leave_parts, phase_parts
gc.collect()

print("[INFO] Raw shapes:", {
    "landing": landing.shape,
    "pos": pos.shape,
    "gs": gs.shape,
    "veh_leave": veh_leave.shape,
    "phase": phase.shape,
})

# =========================================================
# 4) 기본 전처리 (데이터 타입 변환 / 키 정리)
# =========================================================
def prep_common(df, account_col=None):
    df = df.copy()
    if df.empty:
        return df
    if "_D" in df.columns:
        df["_D"] = pd.to_datetime(df["_D"], errors="coerce", utc=True)
    if "matchId" in df.columns:
        df["matchId"] = df["matchId"].astype(str)
    if account_col and account_col in df.columns:
        df = df.rename(columns={account_col: "accountId"})
        df["accountId"] = df["accountId"].astype(str)
    return df

landing = prep_common(landing, account_col="character_accountId")
pos = prep_common(pos, account_col="character_accountId")
gs = prep_common(gs)
veh_leave = prep_common(veh_leave, account_col="character_accountId")
phase = prep_common(phase)

# 결측 제거
if not landing.empty:
    landing = landing.dropna(subset=["matchId", "accountId", "_D", "character_location_x", "character_location_y"]).copy()
if not pos.empty:
    pos = pos.dropna(subset=["matchId", "accountId", "_D", "character_location_x", "character_location_y"]).copy()
if not gs.empty:
    gs = gs.dropna(subset=["matchId", "_D", "gameState_safetyZonePosition_x", "gameState_safetyZonePosition_y", "gameState_safetyZoneRadius"]).copy()
if not veh_leave.empty:
    veh_leave = veh_leave.dropna(subset=["matchId", "accountId", "_D"]).copy()
if not phase.empty:
    phase = phase.dropna(subset=["matchId", "_D"]).copy()

def to_bool(s: pd.Series) -> pd.Series:
    if str(s.dtype) in ("bool", "boolean"):
        return s.astype("boolean")
    return (
        s.astype(str).str.lower()
         .map({"true": True, "false": False, "1": True, "0": False, "nan": pd.NA, "none": pd.NA})
         .astype("boolean")
    )

if not pos.empty and "character_isInVehicle" in pos.columns:
    pos["character_isInVehicle"] = to_bool(pos["character_isInVehicle"])
if not pos.empty and "character_isInBlueZone" in pos.columns:
    pos["character_isInBlueZone"] = to_bool(pos["character_isInBlueZone"])

print("[INFO] After prep shapes:", {
    "landing": landing.shape,
    "pos": pos.shape,
    "gs": gs.shape,
    "veh_leave": veh_leave.shape,
    "phase": phase.shape,
})

# =========================================================
# 5) 기준 테이블: landing_first (matchId, accountId 1행)
# =========================================================
if landing.empty:
    raise RuntimeError("landing(LogParachuteLanding) 데이터가 비어 있습니다. 대상 맵/날짜 범위를 확인하세요.")

landing_first = (
    landing.sort_values(["matchId", "accountId", "_D"], kind="mergesort")
          .groupby(["matchId", "accountId"], as_index=False)
          .first()
          .copy()
)

landing_first = landing_first.rename(columns={
    "character_location_x": "drop_x",
    "character_location_y": "drop_y",
    "character_location_z": "drop_z",
    "_D": "drop_time",
    "distance": "parachute_distance"
})

# =========================================================
# 6) Feature: drop_distance_from_path
#    (초반 position으로 비행기 경로 직선 추정 -> 착지점까지 수직거리)
# =========================================================
pos_early = pos[pos["elapsedTime"].notna()].copy() if ("elapsedTime" in pos.columns and not pos.empty) else pd.DataFrame()
if not pos_early.empty:
    pos_early = pos_early[pos_early["elapsedTime"] <= 30]

def fit_flight_line_and_drop_distance(pos_early_df, landing_df):
    out = []
    if pos_early_df.empty or landing_df.empty:
        return pd.DataFrame(columns=["matchId", "accountId", "drop_distance_from_path"])

    for mid, gpos in pos_early_df.groupby("matchId", sort=False):
        gdrop = landing_df[landing_df["matchId"] == mid]
        if gdrop.empty or len(gpos) < 2:
            continue

        x = gpos["character_location_x"].to_numpy(dtype=float)
        y = gpos["character_location_y"].to_numpy(dtype=float)

        # 수직선 문제 회피
        try:
            if np.nanstd(x) >= np.nanstd(y):
                a, b = np.polyfit(x, y, 1)  # y = ax + b
                dx = gdrop["drop_x"].to_numpy(dtype=float)
                dy = gdrop["drop_y"].to_numpy(dtype=float)
                dist = np.abs(a * dx - dy + b) / np.sqrt(a*a + 1.0)
            else:
                a, b = np.polyfit(y, x, 1)  # x = ay + b
                dx = gdrop["drop_x"].to_numpy(dtype=float)
                dy = gdrop["drop_y"].to_numpy(dtype=float)
                dist = np.abs(a * dy - dx + b) / np.sqrt(a*a + 1.0)
        except Exception:
            continue

        tmp = gdrop[["matchId", "accountId"]].copy()
        tmp["drop_distance_from_path"] = dist
        out.append(tmp)

    return pd.concat(out, ignore_index=True) if out else pd.DataFrame(columns=["matchId", "accountId", "drop_distance_from_path"])

feat_drop_path = fit_flight_line_and_drop_distance(pos_early, landing_first)

# 메모리 정리
del pos_early
gc.collect()

# =========================================================
# 7) Feature: early_enemy_density (반경 500m 내 착지 유저 수)
# =========================================================
def compute_early_enemy_density(landing_df, radius=500.0):
    rows = []
    if landing_df.empty:
        return pd.DataFrame(columns=["matchId", "accountId", "early_enemy_density"])

    for mid, g in landing_df.groupby("matchId", sort=False):
        coords = g[["drop_x", "drop_y"]].to_numpy(dtype=float)
        accs = g["accountId"].to_numpy()

        if len(coords) == 0:
            continue

        diff = coords[:, None, :] - coords[None, :, :]
        dist = np.sqrt((diff ** 2).sum(axis=2))
        within = (dist <= radius)
        np.fill_diagonal(within, False)

        tmp = pd.DataFrame({
            "matchId": mid,
            "accountId": accs,
            "early_enemy_density": within.sum(axis=1)
        })
        rows.append(tmp)

    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=["matchId", "accountId", "early_enemy_density"])

feat_density = compute_early_enemy_density(landing_first, radius=500.0)

# =========================================================
# 8) Position 기반 피처
#    - vehicle_use_ratio
#    - bluezone_exposure_ratio
#    - altitude_variance / altitude_std
#    - pos_samples
# =========================================================
if not pos.empty:
    feat_pos_basic = (
        pos.groupby(["matchId", "accountId"], as_index=False)
           .agg(
               vehicle_use_ratio=("character_isInVehicle", "mean"),
               bluezone_exposure_ratio=("character_isInBlueZone", "mean"),
               altitude_variance=("character_location_z", "var"),
               altitude_std=("character_location_z", "std"),
               pos_samples=("character_location_z", "size"),
           )
    )
else:
    feat_pos_basic = pd.DataFrame(columns=[
        "matchId","accountId","vehicle_use_ratio","bluezone_exposure_ratio","altitude_variance","altitude_std","pos_samples"
    ])

# =========================================================
# 9) Feature: max_vehicle_distance
#    우선 VehLeave 사용, 없으면 Position 차량탑승 상태로 fallback
# =========================================================
if (not veh_leave.empty) and {"vehicle_location_x", "vehicle_location_y"}.issubset(set(veh_leave.columns)):
    veh_tmp = veh_leave.merge(
        landing_first[["matchId", "accountId", "drop_x", "drop_y"]],
        on=["matchId", "accountId"], how="left"
    )
    dx = veh_tmp["vehicle_location_x"] - veh_tmp["drop_x"]
    dy = veh_tmp["vehicle_location_y"] - veh_tmp["drop_y"]
    veh_tmp["veh_dist_from_drop"] = np.sqrt(dx*dx + dy*dy)

    feat_max_vehicle_dist = (
        veh_tmp.groupby(["matchId", "accountId"], as_index=False)["veh_dist_from_drop"]
               .max()
               .rename(columns={"veh_dist_from_drop": "max_vehicle_distance"})
    )
    del veh_tmp
    gc.collect()
else:
    pos_vehicle = pos[pos["character_isInVehicle"] == True].copy() if not pos.empty else pd.DataFrame()
    if not pos_vehicle.empty:
        pos_vehicle = pos_vehicle.merge(
            landing_first[["matchId", "accountId", "drop_x", "drop_y"]],
            on=["matchId", "accountId"], how="left"
        )
        dx = pos_vehicle["character_location_x"] - pos_vehicle["drop_x"]
        dy = pos_vehicle["character_location_y"] - pos_vehicle["drop_y"]
        pos_vehicle["veh_dist_from_drop"] = np.sqrt(dx*dx + dy*dy)

        feat_max_vehicle_dist = (
            pos_vehicle.groupby(["matchId","accountId"], as_index=False)["veh_dist_from_drop"]
                      .max()
                      .rename(columns={"veh_dist_from_drop": "max_vehicle_distance"})
        )
        del pos_vehicle
        gc.collect()
    else:
        feat_max_vehicle_dist = pd.DataFrame(columns=["matchId","accountId","max_vehicle_distance"])

# =========================================================
# 10) 메모리 절약형 Position ⨝ GameState 집계
#     - safezone_proximity_mean
#     - safezone_edge_ratio
#     - rotation_timing_score (간이 버전)
# =========================================================
def compute_safezone_and_rotation_features_streaming(pos_df, gs_df, tolerance="15s", verbose=False):
    if pos_df.empty or gs_df.empty:
        empty_safe = pd.DataFrame(columns=["matchId","accountId","safezone_proximity_mean","safezone_edge_ratio"])
        empty_rot = pd.DataFrame(columns=["matchId","accountId","rotation_timing_score"])
        return empty_safe, empty_rot

    pos_df = pos_df.dropna(subset=["matchId", "accountId", "_D"]).copy()
    gs_df = gs_df.dropna(subset=["matchId", "_D"]).copy()

    pos_df["matchId"] = pos_df["matchId"].astype(str)
    gs_df["matchId"] = gs_df["matchId"].astype(str)

    # 필요한 컬럼만 유지 (object 컬럼 최소화)
    pos_keep = [
        "matchId", "accountId", "_D", "elapsedTime",
        "character_location_x", "character_location_y", "character_location_z"
    ]
    pos_keep = [c for c in pos_keep if c in pos_df.columns]
    pos_df = pos_df[pos_keep].copy()

    gs_keep = [
        "matchId", "_D",
        "gameState_safetyZonePosition_x", "gameState_safetyZonePosition_y", "gameState_safetyZoneRadius"
    ]
    gs_keep = [c for c in gs_keep if c in gs_df.columns]
    gs_df = gs_df[gs_keep].copy()

    pos_df = pos_df.sort_values(["matchId","_D"], kind="mergesort").reset_index(drop=True)
    gs_df = gs_df.sort_values(["matchId","_D"], kind="mergesort").reset_index(drop=True)

    gs_groups = {
        mid: g.drop(columns=["matchId"]).sort_values("_D", kind="mergesort").reset_index(drop=True)
        for mid, g in gs_df.groupby("matchId", sort=False)
    }

    safe_parts = []
    rot_parts = []

    pos_groups = pos_df.groupby("matchId", sort=False)
    total_groups = pos_df["matchId"].nunique()

    for idx, (mid, gpos) in enumerate(pos_groups, start=1):
        if verbose and (idx % 200 == 0 or idx == 1):
            print(f"[ASOF] {idx}/{total_groups} matches processed")

        ggs = gs_groups.get(mid)
        if ggs is None or ggs.empty:
            continue

        gpos = gpos.sort_values("_D", kind="mergesort").reset_index(drop=True)

        try:
            merged = pd.merge_asof(
                gpos,
                ggs,
                on="_D",
                direction="backward",
                allow_exact_matches=True,
                tolerance=pd.Timedelta(tolerance)
            )
        except Exception:
            continue

        req = ["gameState_safetyZonePosition_x", "gameState_safetyZonePosition_y", "gameState_safetyZoneRadius"]
        if not all(c in merged.columns for c in req):
            continue

        merged = merged.dropna(subset=req).copy()
        if merged.empty:
            continue

        dx = merged["character_location_x"] - merged["gameState_safetyZonePosition_x"]
        dy = merged["character_location_y"] - merged["gameState_safetyZonePosition_y"]
        merged["safezone_dist"] = np.sqrt(dx*dx + dy*dy)

        radius = merged["gameState_safetyZoneRadius"].replace(0, np.nan)
        merged["safezone_dist_norm"] = merged["safezone_dist"] / radius
        merged["is_edge"] = merged["safezone_dist_norm"] > 0.8

        # safezone 피처
        safe = (
            merged.groupby(["matchId","accountId"], as_index=False)
                  .agg(
                      safezone_proximity_mean=("safezone_dist_norm", "mean"),
                      safezone_edge_ratio=("is_edge", "mean"),
                  )
        )
        safe_parts.append(safe)

        # rotation_timing_score (간이 버전)
        rot_rows = []
        for (mmid, aid), g in merged.sort_values(["accountId","_D"]).groupby(["matchId","accountId"], sort=False):
            d = g["safezone_dist_norm"].to_numpy(dtype=float)
            if len(d) < 3 or np.all(np.isnan(d)):
                rot_rows.append([mmid, aid, np.nan])
                continue

            if "elapsedTime" in g.columns and g["elapsedTime"].notna().sum() > 1:
                t = g["elapsedTime"].to_numpy(dtype=float)
            else:
                t = (g["_D"] - g["_D"].min()).dt.total_seconds().to_numpy(dtype=float)

            dd = np.diff(d)
            valid_idx = np.where(dd < -0.02)[0]
            if len(valid_idx) == 0:
                score = np.nan
            else:
                start_t = t[valid_idx[0] + 1]
                t_min = np.nanmin(t)
                t_max = np.nanmax(t)
                if np.isfinite(t_min) and np.isfinite(t_max) and t_max > t_min:
                    score = (start_t - t_min) / (t_max - t_min)
                else:
                    score = np.nan

            rot_rows.append([mmid, aid, score])

        rot = pd.DataFrame(rot_rows, columns=["matchId","accountId","rotation_timing_score"])
        rot_parts.append(rot)

        # 즉시 해제
        del merged, safe, rot
        if idx % 200 == 0:
            gc.collect()

    feat_safezone = pd.concat(safe_parts, ignore_index=True) if safe_parts else pd.DataFrame(
        columns=["matchId","accountId","safezone_proximity_mean","safezone_edge_ratio"]
    )
    feat_rotation = pd.concat(rot_parts, ignore_index=True) if rot_parts else pd.DataFrame(
        columns=["matchId","accountId","rotation_timing_score"]
    )

    return feat_safezone, feat_rotation

feat_safezone, feat_rotation = compute_safezone_and_rotation_features_streaming(
    pos, gs, tolerance=ASOF_TOLERANCE, verbose=VERBOSE
)

# =========================================================
# 11) 최종 features_match_user 결합
# =========================================================
base_cols = ["matchId", "accountId", "drop_time", "drop_x", "drop_y", "drop_z", "parachute_distance"]
meta_cols = [c for c in ["src_platform", "src_date", "src_map", "src_file_matchid"] if c in landing_first.columns]

features_match_user = landing_first[base_cols + meta_cols].copy()

for feat_df in [
    feat_drop_path,
    feat_density,
    feat_pos_basic,
    feat_max_vehicle_dist,
    feat_safezone,
    feat_rotation,
]:
    features_match_user = features_match_user.merge(feat_df, on=["matchId", "accountId"], how="left")

# =========================================================
# 12) EDA용 뷰 (핵심 피처만)
# =========================================================
requested_cols = [
    "matchId", "accountId", "src_platform", "src_date", "src_map", "drop_time",
    "drop_distance_from_path",
    "early_enemy_density",
    "rotation_timing_score",
    "vehicle_use_ratio",
    "bluezone_exposure_ratio",
    "safezone_proximity_mean",
    "safezone_edge_ratio",
    "altitude_variance",
    "altitude_std",
    "max_vehicle_distance",
    "pos_samples",
]
requested_cols = [c for c in requested_cols if c in features_match_user.columns]
features_view = features_match_user[requested_cols].copy()

# =========================================================
# 13) 결과 확인
# =========================================================
print("[RESULT] features_match_user shape:", features_match_user.shape)
print("[RESULT] features_view shape:", features_view.shape)

print("\n[RESULT] null count (selected features):")
print(features_view.isna().sum().sort_values(ascending=False))

# drop_time 날짜 타입 보정(혹시 merge 과정에서 object면)
if "drop_time" in features_match_user.columns:
    features_match_user["drop_time"] = pd.to_datetime(features_match_user["drop_time"], errors="coerce", utc=True)

# 날짜 분포 확인 (UTC)
if "drop_time" in features_match_user.columns:
    print("\n[RESULT] drop_time UTC date counts (top 20):")
    print(features_match_user["drop_time"].dt.date.value_counts().sort_index().tail(20))

display(features_view.head(10))

[INFO] Collected files: 3463
[INFO] Example target: {'path': WindowsPath('C:/Users/qkrtl/10th/00_Project/04_final/02_parquet_file/04_temp_parquet_files/temp_parquet_files_kakao_20260212/Erangel_01a4a247-7e5a-4835-a2f7-a3f7c9fb2aac.parquet'), 'platform': 'kakao', 'src_date': '20260212', 'map': 'Erangel', 'file_matchid': '01a4a247-7e5a-4835-a2f7-a3f7c9fb2aac'}
[LOAD] 1/3463 : Erangel_01a4a247-7e5a-4835-a2f7-a3f7c9fb2aac.parquet
[LOAD] 50/3463 : Erangel_70194315-491c-4d6e-ac44-5fb025dcc9f9.parquet
[LOAD] 100/3463 : Erangel_cf979f31-a049-4a45-9c0a-5037d2c09d5d.parquet
[LOAD] 150/3463 : Erangel_366d216c-11db-4d0c-951e-406245876824.parquet
[LOAD] 200/3463 : Erangel_9ce5c91b-0999-449e-aca8-d237928c2448.parquet
[LOAD] 250/3463 : Erangel_05b76e81-d2f4-47a8-b6f1-824cfd3a9c1c.parquet
[LOAD] 300/3463 : Erangel_7bbd76bf-bc15-4d9c-be57-160b17074803.parquet
[LOAD] 350/3463 : Erangel_ff83d105-23cc-4e35-b741-f43856c9ef9b.parquet
[LOAD] 400/3463 : Erangel_6385513a-407a-4a07-8c52-d36113c13fbe.parquet
[LO

KeyboardInterrupt: 